# TrafficGuard — FastAPI Backend
**COMP47250 · Team Software Project · Project P14 · UCD Summer 2026**

Owner: Yashi (FastAPI Backend)

This notebook installs dependencies, loads the trained model, and starts the FastAPI server.

**Run cells top to bottom. Leave Cell 7 running — that's the live server.**

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Model name, classes, status |
| `/predict` | POST | Classify a traffic image |
| `/predict/base64` | POST | Classify a base64 image (frontend) |
| `/attack` | POST | Adversarial attack stub (Day 3) |
| `/defence` | POST | Defence stub (Day 3) |
| `/metrics` | GET | metrics_report.json for charts |
| `/training-log` | GET | train_log.csv as JSON |

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install fastapi uvicorn[standard] python-multipart torch torchvision Pillow --quiet
print('✓ Dependencies installed')

## Cell 2 — Imports

In [ ]:
import io
import json
import csv
import logging
from pathlib import Path
from datetime import datetime
from contextlib import asynccontextmanager
from base64 import b64decode

import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import resnet18
from PIL import Image

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel

print('✓ Imports OK')

## Cell 3 — Configuration

> **Edit `CHECKPOINT_PATH`** if your `best.pt` is saved somewhere else.

In [ ]:
# ── Edit this path to match where your best.pt is saved ──
CHECKPOINT_PATH = Path('../checkpoints/best.pt')
TRAIN_LOG_PATH  = Path('../checkpoints/train_log.csv')
METRICS_PATH    = Path('../checkpoints/metrics_report.json')

DEFAULT_CLASSES = ['Low', 'Medium', 'High']
IMG_SIZE        = 224
MAX_FILE_MB     = 10
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'Device          : {DEVICE}')
print(f'Checkpoint path : {CHECKPOINT_PATH.resolve()}')
print(f'Checkpoint exists: {CHECKPOINT_PATH.exists()}')

## Cell 4 — Model loader

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
logger = logging.getLogger('trafficguard')

class ModelState:
    model        = None
    classes      = DEFAULT_CLASSES
    val_acc      = None
    epoch        = None
    loaded       = False
    load_error   = None
    total_params = 0

state = ModelState()

def build_resnet18(num_classes: int) -> nn.Module:
    """Matches trafficguard_model_v1.ipynb: Dropout(0.3) -> Linear(512, num_classes)"""
    model = resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.fc.in_features, num_classes),
    )
    return model

def load_model():
    if not CHECKPOINT_PATH.exists():
        state.load_error = f'Checkpoint not found at {CHECKPOINT_PATH}. Run model/train.py first.'
        logger.warning(state.load_error)
        return
    try:
        ckpt        = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        classes     = ckpt.get('class_names', DEFAULT_CLASSES)
        net         = build_resnet18(len(classes))
        net.load_state_dict(ckpt['model_state_dict'])
        net.eval().to(DEVICE)
        state.model        = net
        state.classes      = classes
        state.val_acc      = ckpt.get('val_acc')
        state.epoch        = ckpt.get('epoch')
        state.loaded       = True
        state.total_params = sum(p.numel() for p in net.parameters())
        logger.info('Model loaded | epoch=%s | val_acc=%.2f%% | device=%s',
                    state.epoch, (state.val_acc or 0) * 100, DEVICE)
    except Exception as e:
        state.load_error = str(e)
        logger.error('Failed to load model: %s', e)

load_model()
print(f'Model loaded : {state.loaded}')
if state.loaded:
    print(f'Val accuracy : {state.val_acc:.2%}')
    print(f'Epoch        : {state.epoch}')
    print(f'Classes      : {state.classes}')
else:
    print(f'Error        : {state.load_error}')
    print('  -> Server will still start and return stub predictions until best.pt is ready.')

## Cell 5 — Image preprocessing & inference helper

In [ ]:
preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def bytes_to_tensor(raw: bytes) -> torch.Tensor:
    img = Image.open(io.BytesIO(raw)).convert('RGB')
    return preprocess(img).unsqueeze(0).to(DEVICE)

def run_inference(tensor: torch.Tensor) -> dict:
    with torch.no_grad():
        import torch.nn.functional as F
        probs     = F.softmax(state.model(tensor), dim=1)[0]
    probs_dict = {cls: round(probs[i].item(), 4) for i, cls in enumerate(state.classes)}
    top_idx    = probs.argmax().item()
    top_conf   = probs[top_idx].item()
    return {
        'label':          state.classes[top_idx],
        'confidence':     round(top_conf, 4),
        'probabilities':  probs_dict,
        'uncertain':      top_conf < 0.60,
    }

ALLOWED_TYPES = {'image/jpeg', 'image/png', 'image/webp', 'image/bmp'}

def validate_upload(file: UploadFile, raw: bytes):
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(status_code=400,
            detail=f'Unsupported file type {file.content_type}. Send JPEG, PNG, WEBP or BMP.')
    if len(raw) > MAX_FILE_MB * 1024 * 1024:
        raise HTTPException(status_code=413,
            detail=f'File too large. Maximum size is {MAX_FILE_MB} MB.')

print('✓ Preprocessing helpers ready')

## Cell 6 — Build FastAPI app & all endpoints

In [ ]:
app = FastAPI(
    title='TrafficGuard API',
    description='Adversarial Robustness Dashboard — UCD COMP47250 P14',
    version='0.1.0',
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['http://localhost:3000', 'http://localhost:5173',
                   'http://127.0.0.1:3000', 'http://127.0.0.1:5173'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# ── Pydantic schemas ────────────────────────────────────────────────────────
class Base64Request(BaseModel):
    image_b64 : str
    filename  : str = 'upload'

# ── GET /health ─────────────────────────────────────────────────────────────
@app.get('/health', tags=['System'])
async def health():
    return {
        'status'          : 'ok' if state.loaded else 'degraded',
        'model_name'      : 'ResNet-18',
        'classes'         : state.classes,
        'model_loaded'    : state.loaded,
        'device'          : str(DEVICE),
        'val_acc'         : round(state.val_acc, 4) if state.val_acc else None,
        'checkpoint_epoch': state.epoch,
        'timestamp'       : datetime.utcnow().isoformat() + 'Z',
        'error'           : state.load_error,
    }

# ── POST /predict ───────────────────────────────────────────────────────────
@app.post('/predict', tags=['Inference'])
async def predict(file: UploadFile = File(...)):
    raw = await file.read()
    validate_upload(file, raw)
    if not state.loaded:
        logger.warning('Model not loaded — returning stub')
        return {'filename': file.filename, 'label': 'Medium', 'confidence': 0.333,
                'probabilities': {c: 0.333 for c in DEFAULT_CLASSES},
                'uncertain': True, 'timestamp': datetime.utcnow().isoformat() + 'Z'}
    try:
        result = run_inference(bytes_to_tensor(raw))
        logger.info('PREDICT | file=%s | label=%s | conf=%.3f',
                    file.filename, result['label'], result['confidence'])
        return {'filename': file.filename, 'timestamp': datetime.utcnow().isoformat() + 'Z', **result}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ── POST /predict/base64 ────────────────────────────────────────────────────
@app.post('/predict/base64', tags=['Inference'])
async def predict_base64(req: Base64Request):
    if not state.loaded:
        raise HTTPException(status_code=503, detail='Model not loaded yet')
    try:
        result = run_inference(bytes_to_tensor(b64decode(req.image_b64)))
        return {'filename': req.filename, 'timestamp': datetime.utcnow().isoformat() + 'Z', **result}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ── POST /attack (stub — Day 3 Payal) ───────────────────────────────────────
@app.post('/attack', tags=['Attacks'])
async def attack(file: UploadFile = File(...), attack_type: str = 'fgsm', epsilon: float = 0.03):
    raw = await file.read()
    validate_upload(file, raw)
    return {'status': 'stub', 'original_label': 'Medium', 'attacked_label': 'Low',
            'attack_type': attack_type, 'epsilon': epsilon,
            'note': 'Connect to attacks/fgsm.py on Day 3'}

# ── POST /defence (stub — Day 3 Rian) ───────────────────────────────────────
@app.post('/defence', tags=['Defences'])
async def defence(file: UploadFile = File(...), defence_type: str = 'smoothing'):
    raw = await file.read()
    validate_upload(file, raw)
    return {'status': 'stub', 'attacked_label': 'Low', 'defended_label': 'Medium',
            'defence_type': defence_type}

# ── GET /metrics ─────────────────────────────────────────────────────────────
@app.get('/metrics', tags=['Dashboard'])
async def metrics():
    if METRICS_PATH.exists():
        return JSONResponse(content=json.loads(METRICS_PATH.read_text()))
    if state.loaded:
        return {'model': 'ResNet-18', 'classes': state.classes,
                'val_acc': state.val_acc, 'epoch': state.epoch,
                'note': 'Run full evaluation to generate metrics_report.json'}
    raise HTTPException(status_code=404, detail='metrics_report.json not found.')

# ── GET /training-log ────────────────────────────────────────────────────────
@app.get('/training-log', tags=['Dashboard'])
async def training_log():
    if not TRAIN_LOG_PATH.exists():
        raise HTTPException(status_code=404, detail='train_log.csv not found.')
    rows = []
    with open(TRAIN_LOG_PATH, newline='') as f:
        for row in csv.DictReader(f):
            rows.append({k: float(v) if k != 'epoch' else int(float(v)) for k, v in row.items()})
    return JSONResponse(content={'log': rows})

# ── GET /model-info ──────────────────────────────────────────────────────────
@app.get('/model-info', tags=['System'])
async def model_info():
    if not state.loaded:
        raise HTTPException(status_code=503, detail='Model not loaded')
    return {'architecture': 'ResNet-18', 'head': 'Dropout(0.3) -> Linear(512,3)',
            'total_params': state.total_params, 'classes': state.classes,
            'checkpoint_epoch': state.epoch, 'val_acc': state.val_acc, 'device': str(DEVICE)}

print('✓ FastAPI app and all endpoints defined')
print('  Endpoints registered:')
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f'    {list(route.methods)[0]:5s}  {route.path}')

## Cell 7 — Start the server

> ⚠️ **This cell runs forever** — that's normal. The server stays alive until you interrupt it.
>
> After running, open: **http://localhost:8000/docs** to see all endpoints live.
>
> To stop: click the **■ Stop** button or press **Ctrl+C** in the kernel.

In [ ]:
import uvicorn

print('Starting TrafficGuard API server...')
print('  URL  : http://localhost:8000')
print('  Docs : http://localhost:8000/docs')
print('  Stop : Kernel > Interrupt')
print()

uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

## Cell 8 — Quick smoke test (run in a separate kernel / terminal)

While the server is running in Cell 7, open a **new terminal** and run:

```bash
python backend/test_api.py
```

Or test manually with curl:

```bash
# Health check
curl http://localhost:8000/health

# Predict with a real image
curl -X POST http://localhost:8000/predict \
     -F "file=@data/raw/MIO-TCD-Localization/train/00000001.jpg"
```